In [2]:
import os
import pandas as pd

In [3]:
def load_dataset(base_dir, excel_name, audio_folder_name, result_name):
    """
    Loads dataset from a base directory (e.g., training_data or testing_data),
    matches audio files with their metadata, and saves results to output CSV.
    """

    # Dynamically locate important paths
    excel_path = os.path.join(base_dir, excel_name)
    audio_root = os.path.join(base_dir, audio_folder_name)
    results_dir = os.path.join(os.path.dirname(__file__), "results") if "__file__" in globals() else "results"

    # Make sure output folder exists
    os.makedirs(results_dir, exist_ok=True)
    results_csv = os.path.join(results_dir, result_name)

    # --- Read Excel ---
    df = pd.read_excel(excel_path)
    df.columns = df.columns.str.strip().str.lower()

    # --- Collect all .wav files recursively ---
    audio_files = []
    for root, _, files in os.walk(audio_root):
        for f in files:
            if f.lower().endswith('.wav'):
                audio_files.append(os.path.join(root, f))

    # --- Match audio files to Excel IDs ---
    data_records = []
    for _, row in df.iterrows():
        file_id = str(row['id']).strip()
        matched = [f for f in audio_files if file_id in os.path.basename(f)]
        for fpath in matched:
            record = {
                'file_path': os.path.abspath(fpath),
                'id': file_id,
                'age': row.get('age', None),
                'sex': row.get('sex', None),
                'class': row.get('class', None) if 'class' in df.columns else None
            }
            data_records.append(record)

    # --- Save to CSV ---
    result_df = pd.DataFrame(data_records)
    result_df.to_csv(results_csv, index=False)
    print(f"{result_name} saved with {len(result_df)} entries at {results_csv}")

    return result_df

In [4]:
# ----------------------------------------------------
# MAIN EXECUTION (works anywhere)
# ----------------------------------------------------
if __name__ == "__main__":

    # Find the project root (the folder containing 'Baseline Pipeline Implementation')
    current_dir = os.getcwd()
    print(f"Running from: {current_dir}")

    # Define relative paths (automatically work on any machine)
    train_dir = os.path.join(current_dir, "..", "training_data")
    test_dir = os.path.join(current_dir, "..", "testing_data")

    # Run both loaders
    print("\nLoading training dataset...")
    train_df = load_dataset(train_dir, "sand_task_1.xlsx", "training", "train_data.csv")

    print("\nLoading testing dataset...")
    test_df = load_dataset(test_dir, "sand_task1_test.xlsx", "test", "test_data.csv")

Running from: d:\University\AI\SAND Project\Detecting_Dysarthia\Baseline Pipeline Implementation

Loading training dataset...
train_data.csv saved with 2176 entries at results\train_data.csv

Loading testing dataset...
test_data.csv saved with 784 entries at results\test_data.csv
